#    多首歌曲爬蟲任務**


1.   直接在colab的環境中打開雲端資料夾spotify_chart_data中收集到的多個csv檔，依序完成擷取track_ID，移除重複項track_ID後，存成track_id的清單(global、Japan、Korean各一份)。
2.   呼叫上述清單，並為取得的多項track_ID去Spotify API爬取補充資訊。
3.   將補充資訊整理好，生出csv檔存在google drive上。

第一步: 直接在colab的環境中打開雲端資料夾spotify_chart_data中收集到的多個csv檔，依序完成擷取track_ID，移除重複項track_ID後，存成track_id的清單(global、Japan、Korean各一份)。

由於萃取track_id的程式碼會被寫在N02那份ipynb，所以需要了解怎麼在colab上import套件

In [ ]:
from pathlib import Path # 用來為API數據爬取後做存檔
from google.colab import drive # 掛載專案雲端資料夾，該資料夾存放本專案的.ipynb、csv檔。
drive.mount('/content/drive')

%run '<google_drive_path_to>/N02_csv_extract_trackID.ipynb' #N02.ipynb

# import成功後，要確定一下是否調用得到該ipynb中的函式與函式回傳值。例如：
print(kr_track_id_set_in_3m) # 此列應成功印出3個月內有進榜的kr_track_id

第二步: 從第一步完成的清單中遍歷track_ID，為每首歌去上Spotify API爬取補充資訊。

In [ ]:
%run '<google_drive_path_to>/N03_API_general_extract.ipynb'

In [ ]:
jp_API_extract_track_info = [] # 存放從API爬下的日本歌曲資料。

for id in jp_track_id_set_in_3m:
    url = f'https://api.spotify.com/v1/tracks/{id}'
    response = (get_spotify_data_track(url, token=get_token(), sleeptime=3)) # 從雲端環境爬容易發生429 error，所以最好改睡久一點，本地端執行這段時sleeptime=0無妨。
    if response is None:
        continue

    data = response.json()  # 官方說回傳的response是json-encoded檔案，所以用json將其轉換成dict

    # 擷取需要的資料(這段for-loop參照到且yt的版本較佳)
    artists = [artist["name"] for artist in data.get("artists", [])]
    available_markets = data.get("available_markets", [])

    jp_API_extract_track_info.append({
        "country_chart": "JP",
        "track_id": id,
        "track_name": data.get("name"),
        "artists": ", ".join(artists),
        "duration_ms": data.get("duration_ms"),
        "popularity_global": data.get("popularity"),
        "available_markets": ", ".join(available_markets)
    })
# 檢查一下有沒有資料減損，True代表完整。
print(len(jp_API_extract_track_info) == len(jp_track_id_set_in_3m))

第三步: 將補充資訊整理好，存在google drive上。

整理從API爬到的歌曲資料，可用pandas套件將python dict存成csv檔，也可以用csv 模組存成csv。本專案因在前面已經使用到了pandas，所以乾脆繼續使用pandas。

In [ ]:
# python dict轉成pandas的dataframe型態，此步驟會決定csv檔在本地端excel打開時的排版。
df = pandas.DataFrame(jp_API_extract_track_info)

# 存成csv。
save_dir = Path('<google_drive_path_to>/02_Data/02_Spotify_API_raw')
save_file = Path(save_dir, 'jp_API_data_20260107.csv')
if save_file.exists():
    print('有同名檔案，存檔失敗')
else:
    df.to_csv(save_file, header=True, index=True, encoding='utf-8-sig')
    print('存檔成功！請至雲端硬碟上檢視。')

In [ ]:
kr_API_extract_track_info = [] # 存放從API爬下的韓國歌曲資料。

for id in kr_track_id_set_in_3m:
    url = f'https://api.spotify.com/v1/tracks/{id}'
    response = (get_spotify_data_track(url, token=get_token(), sleeptime=3)) # 從雲端環境爬容易發生429 error，所以最好改睡久一點，本地端執行這段時sleeptime=0無妨。
    if response is None:
        continue

    data = response.json()  # 官方說回傳的response是json-encoded檔案，所以用json將其轉換成dict

    # 擷取需要的資料(這段for-loop參照到且yt的版本較佳)
    artists = [artist["name"] for artist in data.get("artists", [])]
    available_markets = data.get("available_markets", [])

    kr_API_extract_track_info.append({
        "country_chart": "JP",
        "track_id": id,
        "track_name": data.get("name"),
        "artists": ", ".join(artists),
        "duration_ms": data.get("duration_ms"),
        "popularity_global": data.get("popularity"),
        "available_markets": ", ".join(available_markets)
    })
# 檢查一下有沒有資料減損，True代表完整。
print(len(kr_API_extract_track_info) == len(kr_track_id_set_in_3m))

In [ ]:
# python dict轉成pandas的dataframe型態，此步驟會決定csv檔在本地端excel打開時的排版。
df = pandas.DataFrame(kr_API_extract_track_info)

# 存成csv。
save_dir = Path('<google_drive_path_to>/02_Data/02_Spotify_API_raw')
save_file = Path(save_dir, 'kr_API_data_20260107.csv')
if save_file.exists():
    print('有同名檔案，存檔失敗')
else:
    df.to_csv(save_file, header=True, index=True, encoding='utf-8-sig')
    print('存檔成功！請至雲端硬碟上檢視。')

In [ ]:
global_API_extract_track_info = [] # 存放從API爬下的global歌曲資料。

for id in global_track_id_set_in_3m:
    url = f'https://api.spotify.com/v1/tracks/{id}'
    response = (get_spotify_data_track(url, token=get_token(), sleeptime=3)) # 從雲端環境爬容易發生429 error，所以最好改睡久一點，本地端執行這段時sleeptime=0無妨。
    if response is None:
        continue

    data = response.json()  # 官方說回傳的response是json-encoded檔案，所以用json將其轉換成dict

    # 擷取需要的資料(這段for-loop參照到且yt的版本較佳)
    artists = [artist["name"] for artist in data.get("artists", [])]
    available_markets = data.get("available_markets", [])

    global_API_extract_track_info.append({
        "country_chart": "JP",
        "track_id": id,
        "track_name": data.get("name"),
        "artists": ", ".join(artists),
        "duration_ms": data.get("duration_ms"),
        "popularity_global": data.get("popularity"),
        "available_markets": ", ".join(available_markets)
    })
# 檢查一下有沒有資料減損，True代表完整。
print(len(global_API_extract_track_info) == len(global_track_id_set_in_3m))

In [ ]:
# python dict轉成pandas的dataframe型態，此步驟會決定csv檔在本地端excel打開時的排版。
df = pandas.DataFrame(global_API_extract_track_info)

# 存成csv。
save_dir = Path('<google_drive_path_to>/02_Data/02_Spotify_API_raw')
save_file = Path(save_dir, 'global_API_data_20260107.csv')
if save_file.exists():
    print('有同名檔案，存檔失敗')
else:
    df.to_csv(save_file, header=True, index=True, encoding='utf-8-sig')
    print('存檔成功！請至雲端硬碟上檢視。')

# 儲存完畢後可以進行資料分析與視覺化